# Validator

The model may never type a number. Every number in its text must be a pointer token
(`{ev:cmp2.n_actual}`) that code fills in. This notebook checks that: it removes the
pointer tokens, then every known identifier (series, stores, rows, calls, cases, model
versions, the snapshot), and reports any word that still contains a digit.

Identifiers contain digits (`CA_2_FOODS`, `synthetic_v3`), which is why a plain digit
search is not enough. Days are numbers too: "day 177" fails; a day must come from a
tool field. Numbers spelled out in words ("four") are not caught; the rubric covers
those.

In [ ]:
#| default_exp validator

In [ ]:
#| export
import re
from collections.abc import Iterable

from agentic_forecast_review.generator import (
    Snapshot,
)
from agentic_forecast_review.schemas import (
    POINTER_TOKEN,
    Case,
    Packet,
)

## Stray digits in one text

Identifiers are removed only as whole words, so `CA_2_FOODS` is allowed but
`CA_2_FOODS9` is not. The longest identifiers go first, so `CA_2_FOODS` is not cut
down to `CA_2` before it is matched.

In [ ]:
#| export
_HAS_DIGIT = re.compile(r"\S*\d\S*")


def stray_digits(text: str, allowed: Iterable[str]) -> list[str]:
    """The words in ``text`` that still contain a digit once pointer tokens and known
    identifiers are removed. An empty list means the text is clean."""
    remaining = POINTER_TOKEN.sub(" ", text)
    for identifier in sorted(set(allowed), key=len, reverse=True):
        pattern = rf"(?<![\w-]){re.escape(identifier)}(?![\w-])"
        remaining = re.sub(pattern, " ", remaining)
    return [word.strip(".,;:!?()\"'") for word in _HAS_DIGIT.findall(remaining)]

## The identifiers a packet may use

Everything with a name in the case: its series and stores, the snapshot's row IDs and
model versions, the case and snapshot IDs, and the calls and returned rows in the
packet's own trace.

In [ ]:
#| export
def known_identifiers(case: Case, snapshot: Snapshot, packet: Packet) -> set[str]:
    """Identifiers that may appear in the packet's text."""
    known = {case.case_id, case.family_id, snapshot.snapshot_id}
    for actual in snapshot.actuals:
        known |= {actual.row_id, actual.series_id, actual.store}
    for forecast in snapshot.forecasts:
        known |= {forecast.row_id, forecast.model_version}
    for step in packet.trace:
        if step.result is not None:
            known |= {step.result.call_id, *step.result.row_ids}
    return known

## Every free-text part of a packet

Claim templates, what is not established, and each step's summary. The result maps
each place to its stray words; an empty result means the packet passes.

In [ ]:
#| export
def packet_stray_digits(packet: Packet, allowed: Iterable[str]) -> dict[str, list[str]]:
    """Stray digits per free-text place in the packet, such as ``claims[0]``."""
    known = set(allowed)
    texts = {f"claims[{i}]": claim.template for i, claim in enumerate(packet.claims)}
    texts |= {
        f"not_established[{i}]": text for i, text in enumerate(packet.not_established)
    }
    texts |= {
        f"trace[{i}].summary": step.summary for i, step in enumerate(packet.trace)
    }
    found = {place: stray_digits(text, known) for place, text in texts.items()}
    return {place: words for place, words in found.items() if words}

## Examples

Tokens and identifiers pass.

In [ ]:
allowed = {"CA_2_FOODS", "CA_2", "synthetic_v3", "dq3", "act__CA_2_FOODS__d0175"}
ok = "{ev:dq3.missing_dates} actual dates are missing for CA_2_FOODS (synthetic_v3)."
print(stray_digits(ok, allowed))
assert stray_digits(ok, allowed) == []
assert stray_digits("See act__CA_2_FOODS__d0175.", allowed) == []

Typed numbers fail: a count, a day, a factor.

In [ ]:
for text in (
    "4 dates are missing.",
    "The gap starts on day 177.",
    "Forecasts fell to 0.85.",
):
    print(stray_digits(text, allowed), "<-", text)
assert stray_digits("4 dates are missing.", allowed) == ["4"]
assert stray_digits("The gap starts on day 177.", allowed) == ["177"]
assert stray_digits("Forecasts fell to 0.85.", allowed) == ["0.85"]

A whole packet: the worked example from `03_schemas` passes; the same packet with one
typed number reports where it is.

In [ ]:
from agentic_forecast_review.schemas import (
    CheckQualityArgs,
    Claim,
    ClaimLabel,
    Cost,
    DayRange,
    TerminalState,
    ToolCall,
    ToolResult,
    TraceStep,
)

period = DayRange(start_day=169, end_day=196)
trace = (
    TraceStep(
        turn=1,
        summary="check the data for CA_2_FOODS",
        call=ToolCall(
            call_id="dq1", args=CheckQualityArgs(series=("CA_2_FOODS",), days=period)
        ),
        result=ToolResult(
            call_id="dq1",
            ok=True,
            fields={"ca_2_foods__missing_days": 4},
            result_hash="a" * 64,
        ),
    ),
    TraceStep(turn=2, summary="finish", observation_refs=("dq1",)),
)


def packet_with(template: str) -> Packet:
    return Packet(
        case_id="fam_01__c2",
        runner="agent",
        snapshot_id="snap_x",
        terminal_state=TerminalState.STOP_UNTRUSTWORTHY,
        claims=(
            Claim(
                label=ClaimLabel.FINDING,
                template=template,
                row_refs=("act__CA_2_FOODS__d0175",),
            ),
        ),
        trace=trace,
        cost=Cost(turns=2, calls=1, usd=0.0, seconds=1.0),
    )


good = packet_with("{ev:dq1.ca_2_foods__missing_days} actual days are missing.")
bad = packet_with(
    "{ev:dq1.ca_2_foods__missing_days} actual days are missing, from day 175."
)
print(packet_stray_digits(good, allowed), packet_stray_digits(bad, allowed))
assert packet_stray_digits(good, allowed) == {}
assert packet_stray_digits(bad, allowed) == {"claims[0]": ["175"]}